# Machine Learning Assignment 2: NASA Near-Earth Objects (NEO) Hazard Classification
**M.Tech (AIML) - Machine Learning**

---

## 1. Problem Statement & Dataset Description

### Problem Statement
Near-Earth Objects (NEOs) are comets and asteroids nudged by the gravitational attraction of nearby planets into orbits that allow them to enter the Earth's neighborhood. The objective of this project is to build and evaluate **5 distinct machine learning classification models** to predict whether a Near-Earth Object is a **Potentially Hazardous Asteroid (PHA)** (`is_potentially_hazardous_asteroid`) based on orbital parameters, physical measurements, observation history, and close-approach metrics.

### Dataset Overview
* **Dataset**: NASA Near-Earth Objects (NEO) Dataset
* **Source**: [Kaggle Public Dataset](https://www.kaggle.com/datasets/sunethjayawardana/nasa-near-earth-objects-neo-full-dataset?resource=download)
* **File Name**: `neo_dur_df.csv`
* **Target Variable**: `is_potentially_hazardous_asteroid` (`0` = Non-Hazardous, `1` = Potentially Hazardous)
* **Instance Size** = 25869
* **Model Features Used**: 22 quantitative numerical predictor variables (filtered from 44 raw columns)
* **Features Selected (22 Predictor Variables)**:
  1. `data_arc_in_days` - Number of days over which the object was observed
  2. `observations_used` - Total astronomical observations recorded
  3. `orbit_uncertainty` - Orbit uncertainty parameter (0-9)
  4. `minimum_orbit_intersection` - Minimum Orbit Intersection Distance (MOID in AU)
  5. `jupiter_tisserand_invariant` - Tisserand's parameter relative to Jupiter
  6. `epoch_osculation` - Epoch of osculation (Julian Date)
  7. `eccentricity` - Orbital eccentricity
  8. `semi_major_axis` - Semi-major axis (AU)
  9. `inclination` - Orbital inclination (degrees)
  10. `ascending_node_longitude` - Longitude of the ascending node (degrees)
  11. `orbital_period` - Orbital period (days)
  12. `perihelion_distance` - Perihelion distance (AU)
  13. `perihelion_argument` - Argument of perihelion (degrees)
  14. `aphelion_distance` - Aphelion distance (AU)
  15. `perihelion_time` - Time of perihelion passage (Julian Date)
  16. `mean_anomaly` - Mean anomaly (degrees)
  17. `mean_motion` - Mean orbital motion (degrees/day)
  18. `absolute_magnitude_h` - Absolute magnitude ($H$)
  19. `estimated_diameter_kilometers_min` - Minimum estimated diameter (km)
  20. `estimated_diameter_kilometers_max` - Maximum estimated diameter (km)
  21. `miss_distance_kilometers0` - Close approach miss distance (km)
  22. `relative_velocity0` - Relative velocity at close approach (km/s)

In [5]:
# ==============================================================================
# Step 1: Import Libraries and Load Dataset (neo_dur_df)
# ==============================================================================

import os
import joblib
import pandas as pd
import numpy as np

# Machine Learning & Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

# Evaluation Metrics
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score, recall_score, f1_score, matthews_corrcoef
)

# Create directory to save trained model artifacts
os.makedirs('model', exist_ok=True)

# Dataset location URL:
# https://github.com/2026aj05209-ayan/MLAssigment2/blob/main/model/source_dataset/neo_dur_df.csv

# Load dataset named 'neo_dur_df.csv' into DataFrame 'neo_dur_df'
neo_dur_df = pd.read_csv('https://raw.githubusercontent.com/2026aj05209-ayan/MLAssigment2/main/model/source_dataset/neo_dur_df.csv')

print("=== Dataset Information ===")
print(f"Total Rows: {neo_dur_df.shape[0]}")
print(f"Total Columns: {neo_dur_df.shape[1]}")
print("\nFirst 5 Rows Preview:")
display(neo_dur_df.head())

=== Dataset Information ===
Total Rows: 25869
Total Columns: 44

First 5 Rows Preview:


,Unnamed: 0,id,orbit_id,orbit_determination_date,first_observation_date,last_observation_date,data_arc_in_days,observations_used,orbit_uncertainty,minimum_orbit_intersection,...,relative_velocity0,orbiting_body1,miss_distance_kilometers1,relative_velocity1,orbiting_body2,miss_distance_kilometers2,relative_velocity2,orbiting_body3,miss_distance_kilometers3,relative_velocity3
0,0,2000433,659,2021-05-24 17:55:05,1893-10-29,2021-05-13,46582.0,9130,0,0.148353,...,5.578619,Earth,7.053323e+07,4.394491,Earth,7.468781e+07,4.816784,Earth,5.382329e+07,4.596055
1,1,2000719,272,2025-03-28 06:20:45,1911-10-04,2025-03-28,41449.0,2102,0,0.201318,...,3.446029,Juptr,2.119996e+08,3.263461,Juptr,2.223920e+08,3.724128,Juptr,2.930859e+08,5.004843
2,2,2001036,1392,2025-04-29 06:20:34,1924-10-23,2025-04-18,36702.0,12602,0,0.343339,...,6.303866,Juptr,2.928564e+08,6.672872,Mars,1.431039e+07,14.980296,Juptr,2.931775e+08,6.220393
3,3,2001221,143,2025-02-20 05:51:07,1932-03-12,2025-02-19,33947.0,686,0,0.107969,...,10.770174,Earth,2.026005e+07,9.557531,Earth,1.738689e+07,8.855631,Earth,1.613093e+07,8.394932
4,4,2001566,194,2025-08-19 08:22:05,1949-07-01,2025-08-18,27807.0,1792,0,0.033511,...,27.008056,Merc,1.486854e+07,42.897635,Earth,8.564701e+06,30.201259,Venus,1.992676e+07,30.445387


## 2. Feature Selection & Data Preprocessing
In this step, we drop metadata / non-numeric text columns (`id`, `name`, `orbit_class_description`, `equinox`, dates, etc.) and isolate our 22 numerical predictors along with the binary target column `is_potentially_hazardous_asteroid`.

In [3]:
# ==============================================================================
# Step 2: Feature Selection, Target Encoding, and Train-Test Split
# ==============================================================================

# Define list of quantitative predictor features (22 features)
feature_cols = [
    'data_arc_in_days', 'observations_used', 'orbit_uncertainty',
    'minimum_orbit_intersection', 'jupiter_tisserand_invariant',
    'epoch_osculation', 'eccentricity', 'semi_major_axis',
    'inclination', 'ascending_node_longitude', 'orbital_period',
    'perihelion_distance', 'perihelion_argument', 'aphelion_distance',
    'perihelion_time', 'mean_anomaly', 'mean_motion',
    'absolute_magnitude_h', 'estimated_diameter_kilometers_min',
    'estimated_diameter_kilometers_max', 'miss_distance_kilometers0',
    'relative_velocity0'
]

target_col = 'is_potentially_hazardous_asteroid'

# Extract features and target from neo_dur_df
X = neo_dur_df[feature_cols].copy()

# Convert target boolean (True/False or 'True'/'False') to binary integers (1/0)
y = neo_dur_df[target_col].astype(int)

# Handle potential missing values by filling with median
X.fillna(X.median(), inplace=True)

# Train / Test Split (80% Train, 20% Test with Stratification)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")

# Save test dataset as required by Step 3 ('test_data.csv')
test_data = pd.concat([X_test, y_test], axis=1)
test_data.to_csv('test_data.csv', index=False)
print("Successfully exported 'test_data.csv' for Streamlit inference!")

X_train shape: (20695, 22), X_test shape: (5174, 22)
Successfully exported 'test_data.csv' for Streamlit inference!


## 3. Feature Scaling & Model Training
We apply `StandardScaler` to normalize feature distributions. Distance-sensitive algorithms (Logistic Regression, KNN, Naive Bayes, SVM) use scaled data, while Decision Trees use raw inputs.

In [4]:
# ==============================================================================
# Step 3: Feature Scaling & Training 5 Classification Models
# ==============================================================================

# Fit StandardScaler on Training Data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Save scaler artifact for Streamlit deployment
joblib.dump(scaler, 'model/scaler.pkl')

# Initialize 5 Classification Models
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=6),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Naive Bayes': GaussianNB(),
    'Support Vector Machine': SVC(probability=True, random_state=42)
}

# Training and Evaluation Loop
results = []

for name, model in models.items():
    # Use scaled data for distance-based / linear models
    if name in ['Logistic Regression', 'KNN', 'Naive Bayes', 'Support Vector Machine']:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_proba = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]

    # Calculate required evaluation metrics
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    mcc = matthews_corrcoef(y_test, y_pred)

    results.append({
        'ML Model Name': name,
        'Accuracy': round(acc, 4),
        'AUC': round(auc, 4),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1': round(f1, 4),
        'MCC': round(mcc, 4)
    })

    # Export trained model artifact to model/ directory
    model_filename = name.lower().replace(' ', '_') + '.pkl'
    joblib.dump(model, os.path.join('model', model_filename))

# Display Metrics Comparison Table
results_df = pd.DataFrame(results)
print("\n=== MODEL EVALUATION METRICS COMPARISON TABLE ===")
display(results_df)


=== MODEL EVALUATION METRICS COMPARISON TABLE ===


,ML Model Name,Accuracy,AUC,Precision,Recall,F1,MCC
0,Logistic Regression,0.9679,0.9900,0.8450,0.818,0.8313,0.8137
1,Decision Tree,0.9973,0.9956,0.9880,0.984,0.9860,0.9845
2,KNN,0.9322,0.9334,0.6987,0.524,0.5989,0.5696
3,Naive Bayes,0.9509,0.9781,0.7189,0.808,0.7608,0.7351
4,Support Vector Machine,0.9710,0.9922,0.8755,0.816,0.8447,0.8294


## 4. Model Comparison Table & Observations

### Evaluation Metrics Comparison Table
| ML Model Name | Accuracy | AUC | Precision | Recall | F1 | MCC |
|---|:---:|:---:|:---:|:---:|:---:|:---:|
| **Logistic Regression** | 0.9679 | 0.9900 | 0.8450 | 0.8180 | 0.8313 | 0.8137 |
| **Decision Tree** | **0.9973** | **0.9956** | **0.9880** | **0.9840** | **0.9860** | **0.9845** |
| **KNN** | 0.9322 | 0.9334 | 0.6987 | 0.5240 | 0.5989 | 0.5696 |
| **Naive Bayes** | 0.9509 | 0.9781 | 0.7189 | 0.8080 | 0.7608 | 0.7351 |
| **Support Vector Machine** | 0.9710 | 0.9922 | 0.8755 | 0.8160 | 0.8447 | 0.8294 |

### Observations on Model Performance
| ML Model Name | Observation about model performance |
|---|---|
| **Logistic Regression** | Achieves strong accuracy (**96.79%**) and AUC (**0.9900**), proving that orbital dimensions provide strong linear separation when scaled. |
| **Decision Tree** | Outperforms all other models across all 6 metrics (**99.73% Accuracy**, **0.9956 AUC**, **0.9860 F1**). Its non-linear decision boundary effectively captures strict physical asteroid hazard rules (e.g., MOID $\le 0.05$ AU and diameter $\ge 140$ m). |
| **KNN** | Delivers lower performance across precision (**69.87%**) and recall (**52.40%**) due to distance distortion across 22 feature dimensions (curse of dimensionality). |
| **Naive Bayes** | Provides higher recall (**80.80%**) than precision (**71.89%**), making it effective at detecting hazards, though conditional independence assumptions cause false positives. |
| **Support Vector Machine** | Second-best performing model (**97.10% Accuracy**, **0.9922 AUC**) using an RBF kernel to isolate boundary conditions effectively. |
| **Overall Winner for dataset?** | **Decision Tree Classifier**. It achieves the highest Accuracy (0.9973), AUC (0.9956), F1 Score (0.9860), and MCC (0.9845). |